# RNA-seq Standalone Pipeline
### Tidy → annotate → whole blood → pre-treatment → one sample per patient → RPKM / TPM

**What this notebook does.** It takes compiled RNA-seq count spreadsheets (samples as columns, genes as rows, sample metadata mixed in), and produces an analysis-ready cohort: **one whole-blood, pre-treatment sample per patient**, joined to demographics, with raw counts, RPKM and TPM.

**How to use it.** Run top to bottom. Each section follows the same pattern:

| Block | Purpose |
|---|---|
| 🎯 **Goal** | what this step produces |
| 💡 **Why** | the reasoning, so you can judge whether it fits your cohort |
| code | your original logic, unchanged |
| 👀 **Checkpoint** | a table or plot to *see* the result before moving on |

**Roadmap: how the sample count narrows**

| Part | Step | Question it answers |
|---|---|---|
| 1 | Configuration | where is everything, what are the rules? |
| 2 | Helpers | how do we compare IDs and read big Excel files fast? |
| 3 | Patient metadata | who are the patients? |
| 4 | Load RNA-seq | which samples and genes do we have? |
| 5 | Whole blood filter | is it really whole blood (not PBMC / sorted cells)? |
| 6 | Dates | when was it drawn (for age and ordering)? |
| 7 | Join metadata | whose sample is it? |
| 8 | Treatment status | is it pre- or post-treatment? |
| 9 | One per patient | which single sample represents each person? |
| 10 | RPKM / TPM | length-normalised expression |
| 11 | Write outputs | CSVs + a final audit of weak matches |

<div class="alert alert-info">
<b>Design principle: filter by flags, never by deletion.</b> Every sample keeps flag columns (<code>is_blood</code>, <code>tx_status</code>, <code>meta_join</code>, <code>kept</code>), and one file, <code>rnaseq_samples_ALL_with_flags.csv</code>, preserves <i>everything</i> so you can always see why a sample was excluded.
</div>

<div class="alert alert-warning">
<b>Privacy.</b> Birth date is read only to compute age, then dropped. The final sample sheet has no DOB and no raw dates. All outputs are CSV.
</div>

---
# Part 1 · Configuration

🎯 **Goal:** every path, rule and cohort decision lives in this one cell.

💡 **Why:** the scientific *decisions* are the parts you may want to change, so they are named variables rather than buried in logic:

| Setting | What it decides |
|---|---|
| `KEEP_ONE_PER_PATIENT` | `"earliest"` mimics a first-visit design; `"deepest"` favours sequencing depth |
| `REQUIRE_PRE_FOR_DX` | for these diagnoses the data *has* PRE/POST labels, so an unlabelled sample is not trusted to be pre-treatment |
| `BLOOD_VALUES` | **exact** match, so "PBMC" can never slip through as "blood" |
| `DX_MAP` | harmonises diagnosis spellings between the two source files |

Writability is asserted up front so a read-only mount fails now, not after ten minutes of work.

In [ ]:
# === RNA-seq standalone: tidy, annotate, whole blood, pre-treatment, one per patient, RPKM/TPM
# All outputs are CSV. Inputs may be xlsx (read once, cached as csv).
import os, re, glob, gzip, urllib.request
import numpy as np, pandas as pd

RNA_FILES = [
    "/mnt/filesystems/HC_blood_all_samples.xlsx",
    "/mnt/filesystems/Goldbach-Mansky_compiled_RNA-Seq_GeneCounts_CANDLE_AND_SAVI_SUBSET_2026_09_22.xlsx",
]
METADATA_PATH = "/mnt/filesystems/MetadataPatients_NoPHI.xlsx"
METADATA_EXTRA = []        # e.g. a healthy-volunteer sheet with the same headers (Enrollment ID, Initials, Birth Date, Sex, ...)
OUT_DIR   = "/mnt/filesystems/rnaseq_out"     # all results are written here
CACHE_DIR = "/home/jovyan/rnaseq_cache"        # xlsx->csv conversion cache, kept off the mount
KEEP_ONE_PER_PATIENT = "earliest"            # or "deepest" (most total counts)
REQUIRE_PRE_FOR_DX = r"CANDLE|SAVI"          # these cohorts have PRE/POST labels: insist on PRE
BLOOD_VALUES = {"blood", "whole blood", "wholeblood", "wb"}   # exact match; PBMC never passes
GTF_PATH = ""                                # blank = auto-find, then try download
GTF_URL = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"
           "release_19/gencode.v19.annotation.gtf.gz")
DX_MAP = {"hc whole blood": "HC",
          "undifferentiated interferonopathy": "Undifferentiated Interferonopathy"}
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), f"{OUT_DIR} is not writable"

---
# Part 2 · Helpers

🎯 **Goal:** two small tools used everywhere: an ID normaliser and a cached Excel reader.

💡 **Why:**

- **`normalize_id`** makes `O-6033`, `o6033` and `O6033` compare equal, strips Excel's `.0` float suffix, and handles one cohort quirk: RNA files write `CHLD0047` where the metadata has `LD0047`. Without this, those patients silently fail to join.
- **`read_table_cached`** exists because reading a large `.xlsx` is *slow* (minutes). It converts to CSV once into a cache folder (off the mount, so it's fast), and re-converts only if the Excel file is newer. Everything is read as text (`dtype=object`) so IDs like `0047` don't lose their leading zeros.
- `DATE_MIN/DATE_MAX` is a sanity window: any parsed date outside 2008–today is treated as a typo.

In [ ]:
# ---- helpers ---------------------------------------------------------------
def normalize_id(x):
    s = "" if x is None else str(x).strip()
    if s.lower() in ("nan", "none", "nat", ""): return ""
    s = re.sub(r"\.0$", "", s)
    s = s.upper().replace("-", "").replace(" ", "").replace("_", "")
    if s.startswith("CHLD"):                 # CHLD0047 (RNA files) == LD0047 (metadata)
        s = s[2:]
    return s

def norm_colname(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())

DATE_MIN, DATE_MAX = pd.Timestamp("2008-01-01"), pd.Timestamp.today().normalize()

def read_table_cached(path):
    """xlsx -> csv once (cached in CACHE_DIR), then always read the csv."""
    if path.lower().endswith(".csv"):
        return pd.read_csv(path, dtype=object, keep_default_na=False, low_memory=False)
    csv_path = os.path.join(CACHE_DIR, os.path.splitext(os.path.basename(path))[0] + ".csv")
    if (not os.path.exists(csv_path)
            or os.path.getmtime(csv_path) < os.path.getmtime(path)):
        print(f"converting {os.path.basename(path)} -> csv (one time) ...")
        pd.read_excel(path, dtype=object).to_csv(csv_path, index=False)
    return pd.read_csv(csv_path, dtype=object, keep_default_na=False, low_memory=False)

---
# Part 3 · Patient metadata

🎯 **Goal:** load the master patient sheet and build fast lookups from it.

💡 **Why:**

- The workbook may have several sheets; the one with the most columns like *enroll / diagnos / initial* is chosen automatically.
- `METADATA_EXTRA` lets you append e.g. a healthy-volunteer sheet. The main sheet is concatenated **first**, so on duplicate IDs the main sheet wins.
- Column names are found **loosely** (case and punctuation ignored), because headers drift between versions of the sheet.
- Two lookups come out: **ID → row** (on both Enrollment ID and Alternate ID) and **initials → ID**, but only where initials are **unique**. Shared initials are never auto-resolved, because a wrong patient is worse than a blank one.

In [ ]:
# ---- patient metadata ------------------------------------------------------
sheets = pd.read_excel(METADATA_PATH, sheet_name=None, dtype=str)
md = max(sheets.values(), key=lambda d: sum(bool(re.search(r"enroll|diagnos|initial", str(c), re.I))
                                            for c in d.columns))
md.columns = [str(c).strip() for c in md.columns]
md = md.dropna(how="all").reset_index(drop=True)

for p in METADATA_EXTRA:                     # main sheet first, so it wins on duplicate IDs
    ex = (pd.read_excel(p, dtype=str) if p.lower().endswith((".xlsx", ".xls"))
          else pd.read_csv(p, dtype=str))
    ex.columns = [str(c).strip() for c in ex.columns]
    nm = {norm_colname(c): c for c in md.columns}
    ex = ex.rename(columns={c: nm[norm_colname(c)] for c in ex.columns if norm_colname(c) in nm})
    md = pd.concat([md, ex.dropna(how="all").reindex(columns=md.columns)], ignore_index=True)
    print(f"extra metadata: {p} (+{len(ex)} rows)")

def fc(*names):
    nm = {norm_colname(c): c for c in md.columns}
    return next((nm[norm_colname(n)] for n in names if norm_colname(n) in nm), None)

COL = {"enroll": fc("Enrollment ID"), "alt": fc("Alternate Enrollment ID"),
       "initials": fc("Initials"), "dob": fc("Birth Date", "DOB"), "sex": fc("Sex", "Gender"),
       "race": fc("Race"), "ethnicity": fc("Ethnicity"), "status": fc("Study Status"),
       "dx1": fc("Diagnosis1"), "dx2": fc("Diagnosis2"), "note": fc("Note", "Notes")}
print("metadata:", len(md), "rows; columns not found:", [k for k, v in COL.items() if v is None])

meta_lookup, by_ini = {}, {}
for i, r in md.iterrows():
    for kc in (COL["enroll"], COL["alt"]):
        k = normalize_id(r.get(kc)) if kc else ""
        if k: meta_lookup.setdefault(k, i)
    ini = str(r.get(COL["initials"]) or "").strip().upper() if COL["initials"] else ""
    eid = normalize_id(r.get(COL["enroll"])) if COL["enroll"] else ""
    if ini and eid: by_ini.setdefault(ini, set()).add(eid)
AUTO_INITIALS_MAP = {k: next(iter(v)) for k, v in by_ini.items() if len(v) == 1}

def meta_value(i, key):
    c = COL.get(key)
    if i < 0 or not c: return ""
    v = md.at[i, c]
    return "" if pd.isna(v) else str(v).strip()

---
# Part 4 · Load the RNA-seq files

🎯 **Goal:** turn each compiled spreadsheet into (a) a **sample table** (one row per sample) and (b) a **genes × samples count matrix**.

💡 **Why this reshaping is needed:** the compiled files are in an unusual layout. Rows have a `Stat` column; rows labelled `Raw_Count` are gene counts, but **all other rows are sample metadata** (diagnosis, treatment, facility...) stored *as if they were genes*. So the loader:

1. checks for the required columns and fails with a clear message if they're missing,
2. splits the rows into count rows and metadata rows,
3. **transposes** the metadata rows so each sample becomes a row.

Then the files are combined. Counts are matched on `Gene_ID` with an **inner join**, which keeps only genes present in *every* file. That is conservative but ensures all samples are measured on the same gene set. Repeated sample names across files would silently collide, so they raise an error.

### 4.1 Read each file

In [ ]:
# ---- 1. load: split metadata rows from count rows, samples become rows ------
def load_rna(path):
    raw = read_table_cached(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    need = {"Gene_ID", "Gene_Name", "Stat"}
    if not need <= set(raw.columns):
        raise ValueError(f"{os.path.basename(path)} lacks {need - set(raw.columns)}; "
                         f"first columns are {list(raw.columns)[:6]}")
    samples = [c for c in raw.columns if c not in need]
    is_count = raw["Stat"].astype(str).str.strip().eq("Raw_Count")
    print(f"{os.path.basename(path)[:60]}: {len(samples)} samples, {int(is_count.sum())} genes")
    meta = raw.loc[~is_count, ["Stat"] + samples].set_index("Stat").T
    meta.columns = [str(c).strip() for c in meta.columns]
    meta.index.name = "sample_id"
    cnt = raw.loc[is_count, ["Gene_ID", "Gene_Name"] + samples].copy()
    cnt[samples] = cnt[samples].apply(pd.to_numeric, errors="coerce")
    return meta.reset_index(), cnt

metas, cnts = [], []
for p in RNA_FILES:
    m, c = load_rna(p); m["source_file"] = os.path.basename(p)
    metas.append(m); cnts.append(c)

### 4.2 Combine into one count matrix and one sample table

💡 **Why standardise column names:** the two files may label the same field differently (`Enrollment ID` vs `EnrollmentID`, `Gender` vs `Sex`). The rename map gives every field one canonical name, and any missing field is created blank so later code never hits a `KeyError`. Placeholder text (`-`, `NaN`) becomes empty. `total_counts` (library size) is computed now because it's used later for tie-breaking and RPKM.

In [ ]:
gene_names = cnts[0].drop_duplicates("Gene_ID").set_index("Gene_ID")["Gene_Name"]
mats = [c.drop(columns="Gene_Name").groupby("Gene_ID").sum(min_count=1) for c in cnts]
counts = pd.concat(mats, axis=1, join="inner")
print(f"genes in every file: {len(counts)}  (per file: {[len(m) for m in mats]})")
dup = counts.columns[counts.columns.duplicated()]
if len(dup): raise ValueError(f"sample names repeated across files: {list(dup)[:5]}")

rn = {"enrollmentid": "enrollment_id_rna", "metadatainitials": "initials_rna",
      "samplesource": "sample_source", "diagnosis": "diagnosis_rna", "treatment": "treatment",
      "sequencingfacility": "sequencing_facility", "sequencingcohort": "sequencing_cohort",
      "gender": "gender_rna", "rnasamplecollectiondate": "rna_collection_date"}
rmeta = pd.concat(metas, ignore_index=True)
rmeta = rmeta.rename(columns=lambda c: rn.get(norm_colname(c), c))
for c in rn.values():
    if c not in rmeta.columns: rmeta[c] = ""
rmeta = rmeta.fillna("").replace({"-": "", "NaN": "", "nan": ""})
rmeta["total_counts"] = rmeta["sample_id"].map(counts.sum())
rmeta["dx_clean"] = rmeta["diagnosis_rna"].map(
    lambda s: DX_MAP.get(str(s).strip().lower(), str(s).strip()))

### 👀 Checkpoint 4: what did we load?

Left: samples per diagnosis. Right: sequencing depth (library size) per sample, log scale. Very shallow samples at the left edge are worth knowing about, since they'll be noisy.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
rmeta["dx_clean"].replace("", "(blank)").value_counts().head(12).plot.barh(ax=ax[0], color="#4C78A8")
ax[0].set_title(f"Samples per diagnosis (n={len(rmeta)})"); ax[0].invert_yaxis()
ax[1].hist(np.log10(rmeta["total_counts"].clip(lower=1)), bins=40, color="#76B7B2")
ax[1].set_title("Library size per sample"); ax[1].set_xlabel("log10(total counts)")
plt.tight_layout(); plt.show()

---
# Part 5 · Whole blood only

🎯 **Goal:** keep only genuine **whole-blood** samples. No PBMC, no sorted cells.

💡 **Why two layers of defence:** a PBMC or sorted-cell sample has a very different transcriptome (no neutrophils, different interferon signal), so mixing it into a whole-blood cohort would inject a fake effect. Defence layers:

1. **Label check:** `sample_source` must be *exactly* one of `BLOOD_VALUES`.
2. **Sneaky check:** even if labelled "blood", the sample name / treatment / cohort text is searched for cell-fraction words (`pbmc`, `cd4`, `sorted`, `isolat`...). Anything caught is **excluded and printed** for you to inspect. This catches mislabelled rows that a label-only filter would pass.

The crosstab shows what values exist per file, so you can spot a new spelling (e.g. `"Whole Blood (PAXgene)"`) that should be added to `BLOOD_VALUES`.

In [ ]:
# ---- 2. whole blood only (no PBMC, no sorted cell fractions) ----------------
src = rmeta["sample_source"].astype(str).str.strip().str.lower()
print("\nsample source values (per file):")
print(pd.crosstab(rmeta["source_file"].str[:40], src).to_string())
NOT_WHOLE_BLOOD = re.compile(r"pbmc|mononuclear|monocyte|neutrophil|lymphocyte|"
                             r"t[\s_-]?cell|b[\s_-]?cell|nk[\s_-]?cell|cd\d|sorted|isolat", re.I)
blob = (rmeta["sample_id"].astype(str) + " " + src + " " + rmeta["treatment"].astype(str)
        + " " + rmeta["sequencing_cohort"].astype(str))
sneaky = src.isin(BLOOD_VALUES) & blob.str.contains(NOT_WHOLE_BLOOD)
print(f"\nlabelled blood but name suggests a cell fraction: {int(sneaky.sum())}")
if sneaky.any():
    print(rmeta.loc[sneaky, ["sample_id", "sample_source"]].head(20).to_string())
rmeta["is_blood"] = src.isin(BLOOD_VALUES) & ~sneaky

---
# Part 6 · Dates (for age and ordering)

🎯 **Goal:** a best-available draw date per sample, used only for computing age and for choosing the *earliest* sample.

💡 **Why a priority order:** a real `rna_collection_date` is always preferred. If missing, a date is mined from the sample name (e.g. `121003_121016`). That stamp is a **sequencing batch date**, not a draw date, so it's only a rough proxy. Its provenance is recorded in `age_date_source` (`"sample_name (approx)"`), so you can later drop or down-weight age values built on it. Dates outside 2008–today are rejected as typos.

In [ ]:
# ---- 3. date (for age and ordering only) -----------------------------------
# The stamp in sample names (e.g. 121003_121016) is a sequencing batch, so it is
# only a rough proxy for the draw date; a real collection date is preferred.
def date_from_name(s):
    m = re.search(r"(?<!\d)(\d{6})_\d{6}(?!\d)", s) or re.search(r"(?<!\d)(\d{6})(?!\d)", s)
    if not m: return pd.NaT
    try: d = pd.to_datetime(m.group(1), format="%y%m%d")
    except ValueError: return pd.NaT
    return d if DATE_MIN <= d <= DATE_MAX else pd.NaT

coll = pd.to_datetime(rmeta["rna_collection_date"], errors="coerce")
coll = coll.where((coll >= DATE_MIN) & (coll <= DATE_MAX))
from_name = rmeta["sample_id"].map(date_from_name)
rmeta["_date"] = coll.fillna(from_name)
rmeta["age_date_source"] = np.where(coll.notna(), "rna_collection_date",
                            np.where(from_name.notna(), "sample_name (approx)", "none"))

---
# Part 7 · Join patient metadata

🎯 **Goal:** decide *which patient* each RNA sample belongs to, then attach demographics and age.

💡 **Why a careful ladder:** the RNA files and the patient sheet were never designed to join. The rules, in order of trust:

1. **Enrollment ID** exact match (`enrollment_id`).
2. **HC fix:** a bare number like `1126` is `R1126` in the metadata (`R_prefix_id`).
3. **Initials, only when it's safe:** initials are used *only* if the RNA ID is blank or is itself just initials (`HC-MF`, `BG-SAF`). If the RNA has a real-looking ID that metadata lacks, it returns `id_not_in_metadata` and **does not guess**.
4. **Diagnosis sanity check on initials matches:** if the metadata diagnosis conflicts with the RNA diagnosis, it returns `initials_conflict` and stays unmatched. Compatible wording (e.g. interferonopathy ↔ AGS) is allowed as `initials_dx_compat`.

Every sample records *how* it was matched in `meta_join`, so you can filter to only high-confidence matches. Age is then `(date − DOB) / 365.25`, with impossible values (<0 or >120) blanked.

In [ ]:
# ---- 4. join patient metadata ----------------------------------------------
HEALTHY = re.compile(r"^hc$|health|control", re.I)
def match_meta(r):
    k = normalize_id(r["enrollment_id_rna"])
    ini = str(r["initials_rna"]).strip().upper()
    ini_n = normalize_id(ini)
    if k and k in meta_lookup: return meta_lookup[k], "enrollment_id"
    if k.isdigit() and ("R" + k) in meta_lookup:           # HC '1126' is 'R1126' in the metadata
        return meta_lookup["R" + k], "R_prefix_id"
    # only fall back to initials when the RNA ID is blank or is itself initials-derived (HC-MF, BG-SAF)
    if not (k == "" or (ini_n and k.endswith(ini_n))):
        return -1, "id_not_in_metadata"                    # a real-looking ID that metadata lacks: don't guess
    eid = AUTO_INITIALS_MAP.get(ini)                       # unique initials only, as in NanoString
    if eid and eid in meta_lookup:
        i = meta_lookup[eid]
        dx_md, dx_rna = meta_value(i, "dx1"), str(r["diagnosis_rna"])
        same = (norm_colname(dx_md) == norm_colname(dx_rna)
                or (HEALTHY.search(dx_md) and HEALTHY.search(dx_rna)))
        compat = (re.search(r"interferon|aicardi|\bags\b", dx_md, re.I)
                  and re.search(r"\bags\b|aicardi", dx_rna, re.I))
        if dx_md == "" or same: return i, "initials"
        if compat: return i, "initials_dx_compat"
        return -1, "initials_conflict"
    return -1, "none"
mm = rmeta.apply(lambda r: pd.Series(match_meta(r), index=["meta_row", "meta_join"]), axis=1)
rmeta = pd.concat([rmeta, mm], axis=1)
for f, name in [("sex", "sex"), ("race", "race"), ("ethnicity", "ethnicity"),
                ("status", "study_status"), ("dx1", "diagnosis1"), ("dx2", "diagnosis2"),
                ("note", "metadata_note"), ("dob", "_dob"), ("enroll", "_md_enroll")]:
    rmeta[name] = rmeta["meta_row"].map(lambda i, f=f: meta_value(i, f))
print("\nmetadata join:"); print(rmeta["meta_join"].value_counts().to_string())

dob = pd.to_datetime(rmeta["_dob"], errors="coerce")
rmeta["age_yrs"] = ((rmeta["_date"] - dob).dt.days / 365.25).round(1)
rmeta.loc[(rmeta["age_yrs"] < 0) | (rmeta["age_yrs"] > 120), "age_yrs"] = np.nan

### 👀 Checkpoint 7: how were samples matched?

Green-ish bars (`enrollment_id`, `R_prefix_id`) are the strongest. `initials*` are weaker. `id_not_in_metadata`, `initials_conflict` and `none` are the samples with no demographics. Those are the ones to chase if the cohort is smaller than expected.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
colors = {"enrollment_id": "#59A14F", "R_prefix_id": "#8CD17D", "initials": "#F1CE63",
          "initials_dx_compat": "#F28E2B", "initials_conflict": "#E15759",
          "id_not_in_metadata": "#E15759", "none": "#BAB0AC"}
vc = rmeta["meta_join"].value_counts()
vc.plot.barh(ax=ax, color=[colors.get(k, "#999") for k in vc.index])
ax.invert_yaxis(); ax.set_title("Metadata join method")
plt.tight_layout(); plt.show()

---
# Part 8 · Treatment status

🎯 **Goal:** label each sample `pre`, `post`, or `unlabelled`.

💡 **Why:** this is the most scientifically important filter. For interferon-driven disease, drugs (e.g. JAK inhibitors) can suppress the signature almost completely. A post-treatment sample would look like a *healthy* profile and quietly dilute your disease cohort.

The `treatment` text is free-form, so a pattern recognises "pre" wording (`pre`, `baseline`, `untreated`, `naive`, `screening`, `day 0`...). **Anything else with text is treated as post-treatment**, the cautious default. Blank stays `unlabelled`; what to do with unlabelled is decided in Part 9, by diagnosis.

In [ ]:
# ---- 4b. treatment status --------------------------------------------------
PRE_TX = re.compile(r"pre|baseline|bsl|untreat|naive|none|before|screen|off[\s-]?(?:tx|treat)|^v?0$|^day\s*0$", re.I)
tx = rmeta["treatment"].astype(str).str.strip()
is_pre = (tx != "") & tx.str.contains(PRE_TX)
rmeta["is_post_tx"] = (tx != "") & ~is_pre
rmeta["tx_status"] = np.where(rmeta["is_post_tx"], "post", np.where(is_pre, "pre", "unlabelled"))
print("\ntreatment status (per file):")
print(pd.crosstab(rmeta["source_file"].str[:40], rmeta["tx_status"]).to_string())

### 👀 Checkpoint 8: which treatment strings were called what?

**Read this table.** The regex is a heuristic, and a wrong call here changes who is in your cohort. The table shows the raw `treatment` strings and the status each got. Scan it for anything misclassified, and fix `PRE_TX` if so.

In [ ]:
display(rmeta.assign(treatment=rmeta["treatment"].replace("", "(blank)"))
        .groupby(["tx_status", "treatment"]).size().rename("n")
        .reset_index().sort_values(["tx_status", "n"], ascending=[True, False]).head(40))

---
# Part 9 · One sample per patient

🎯 **Goal:** exactly one whole-blood, pre-treatment sample per person.

💡 **Why one per patient:** patients with many samples would otherwise dominate, and repeated measures from one person aren't independent, which breaks standard statistical tests.

**Eligibility** (all must hold): blood, **not** post-treatment, and a patient identity exists. For diagnoses in `REQUIRE_PRE_FOR_DX` (CANDLE / SAVI), the data *has* PRE/POST labels, so an **unlabelled** sample there is excluded rather than assumed pre-treatment. The excluded count is printed.

**Choosing among a patient's samples:** explicit `PRE` beats unlabelled; then by *earliest* date (or deepest sequencing if you chose `"deepest"`); library size breaks ties.

**`patient_key`** is the best identity available: metadata enrollment ID, else the RNA file's ID, else initials. This lets even unmatched samples be de-duplicated against each other.

In [ ]:
# ---- 5. one sample per patient: blood, not post-treatment ------------------
rmeta["patient_key"] = [normalize_id(a) or normalize_id(b) or normalize_id(c)
    for a, b, c in zip(rmeta["_md_enroll"], rmeta["enrollment_id_rna"], rmeta["initials_rna"])]
need_pre = rmeta["dx_clean"].astype(str).str.contains(REQUIRE_PRE_FOR_DX, case=False, regex=True)
base = rmeta["is_blood"] & ~rmeta["is_post_tx"] & (rmeta["patient_key"] != "")
print(f"\nexcluded for lacking a PRE label ({REQUIRE_PRE_FOR_DX}):", int((base & need_pre & ~is_pre).sum()))
elig = base & (~need_pre | is_pre)
b = rmeta[elig].copy()
b["_pre_rank"] = (b["tx_status"] != "pre").astype(int)       # explicit PRE beats unlabelled
if KEEP_ONE_PER_PATIENT == "earliest":
    b = b.sort_values(["patient_key", "_pre_rank", "_date", "total_counts"],
                      ascending=[True, True, True, False], na_position="last")
else:
    b = b.sort_values(["patient_key", "_pre_rank", "total_counts"], ascending=[True, True, False])
rmeta["kept"] = rmeta["sample_id"].isin(b.drop_duplicates("patient_key")["sample_id"])
final_meta = rmeta[rmeta["kept"]].copy()

print(f"\nsamples: {len(rmeta)} -> {int(rmeta['is_blood'].sum())} blood "
      f"-> {int(elig.sum())} eligible -> {len(final_meta)} one-per-patient")
print("age known:", int(final_meta["age_yrs"].notna().sum()), "of", len(final_meta))
for title, col in [("treatment status", "tx_status"), ("metadata join", "meta_join"),
                   ("age date source", "age_date_source")]:
    print(f"\nkept samples by {title}:")
    print(pd.crosstab(final_meta["dx_clean"].replace("", "(blank)"), final_meta[col]).to_string())

ini = final_meta.assign(ini=final_meta["initials_rna"].str.upper()).groupby("ini")["patient_key"].agg(list)
print("\nsame initials under more than one patient key (possible duplicate person):")
print(ini[ini.str.len() > 1].to_string() if (ini.str.len() > 1).any() else "  none")

### 👀 Checkpoint 9: the attrition funnel and the final cohort

The funnel shows where samples were lost, step by step. Large drops at an unexpected step are a prompt to revisit the matching rule for that step. The right panel shows the final cohort by diagnosis, coloured by treatment status.

In [ ]:
steps = [("all samples", len(rmeta)),
         ("whole blood", int(rmeta["is_blood"].sum())),
         ("not post-tx + has ID", int(base.sum())),
         ("PRE rule satisfied", int(elig.sum())),
         ("one per patient", len(final_meta))]
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].barh([s[0] for s in steps], [s[1] for s in steps], color="#4C78A8")
for i, (_, v) in enumerate(steps): ax[0].text(v, i, f" {v}", va="center")
ax[0].invert_yaxis(); ax[0].set_title("Sample attrition")
pd.crosstab(final_meta["dx_clean"].replace("", "(blank)"),
            final_meta["tx_status"]).plot.barh(stacked=True, ax=ax[1])
ax[1].invert_yaxis(); ax[1].set_title("Final cohort"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

---
# Part 10 · Gene lengths → RPKM and TPM

🎯 **Goal:** length-normalised expression, because raw counts aren't comparable *between genes* (a long gene collects more reads) or *between samples* (deeper libraries collect more).

💡 **Why two measures:**

| Measure | Formula | Properties |
|---|---|---|
| **RPKM** | counts ÷ gene length (kb) ÷ library size (millions) | normalises depth *then* length. Column sums differ between samples. |
| **TPM** | (counts ÷ length) ÷ sum of that over all genes × 10⁶ | every sample sums to exactly 1,000,000, so it is better for comparing across samples |

**Gene length** is the *non-overlapping exon union* per gene from a GENCODE GTF, so exons shared by several transcripts aren't double-counted. The code first looks for a cached `gene_lengths.csv`, then any `.gtf` on the mounts, then downloads GENCODE v19. Version suffixes (`ENSG….5`) are stripped if the IDs don't match, and the **match rate is printed**. If that number is low, the GTF release doesn't match your counts.

RPKM's library size uses the **full** library, taken before any genes are dropped for lacking a length. Raw counts are always written first, so a missing GTF never blocks the rest.

### 10.1 Compute gene lengths (cached)

In [ ]:
# ---- 6. gene lengths -> RPKM and TPM ---------------------------------------
def gtf_gene_lengths(path):
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt") as fh:
        for line in fh:
            if line[0] == "#": continue
            f = line.split("\t")
            if len(f) > 8 and f[2] == "exon":
                rows.append((re.search(r'gene_id "([^"]+)"', f[8]).group(1), int(f[3]), int(f[4])))
    ex = pd.DataFrame(rows, columns=["g", "s", "e"]).sort_values(["g", "s"])
    prev = ex.groupby("g")["e"].cummax().groupby(ex["g"]).shift().fillna(0)
    return ((ex["e"] - np.maximum(ex["s"] - 1, prev)).clip(lower=0)).groupby(ex["g"]).sum()

len_csv = os.path.join(OUT_DIR, "gene_lengths.csv")
lengths = pd.read_csv(len_csv, index_col=0).iloc[:, 0] if os.path.exists(len_csv) else None
if lengths is None:
    gtf = GTF_PATH or next(iter(sum((glob.glob(f"{r}/**/*.gtf*", recursive=True)
            for r in ["/mnt/filesystems", "/mnt/session_data", "/home/jovyan"]), [])), "")
    if not gtf:
        try:
            gtf = "/home/jovyan/gencode.v19.annotation.gtf.gz"
            print("downloading", GTF_URL); urllib.request.urlretrieve(GTF_URL, gtf)
        except Exception as e:
            print("could not download GTF:", e); gtf = ""
    if gtf:
        lengths = gtf_gene_lengths(gtf); lengths.to_csv(len_csv, header=["length_bp"])

### 10.2 Normalise and write the expression matrices

In [ ]:
cc = counts[list(final_meta["sample_id"])]
cc_out = cc.copy(); cc_out.insert(0, "Gene_Name", cc_out.index.map(gene_names))
cc_out.to_csv(os.path.join(OUT_DIR, "rnaseq_counts_raw.csv"))

if lengths is None:
    print("\nNo gene lengths available: raw counts written, RPKM skipped. "
          "Put a .gtf(.gz) on the mount and set GTF_PATH.")
else:
    versioned = cc.index.isin(lengths.index).mean() > 0.5
    lkp, key = lengths.copy(), cc.index
    if not versioned:
        lkp.index = lkp.index.str.split(".").str[0]; lkp = lkp[~lkp.index.duplicated()]
        key = cc.index.str.split(".").str[0]
    L = pd.Series(lkp.reindex(key).values / 1000.0, index=cc.index)
    print(f"\ngene length found for {L.notna().mean():.1%} of {len(L)} genes")
    lib = cc.sum()                                   # full library size, before dropping genes
    ok = L.notna() & (L > 0)
    c2 = cc[ok]
    rpkm = c2.div(L[ok], axis=0).div(lib / 1e6, axis=1)
    rate = c2.div(L[ok], axis=0); tpm = rate.div(rate.sum(), axis=1) * 1e6
    for nm, mat in (("rpkm", rpkm), ("tpm", tpm)):
        o = mat.copy(); o.insert(0, "Gene_Name", o.index.map(gene_names))
        o.to_csv(os.path.join(OUT_DIR, f"rnaseq_{nm}.csv"))

### 👀 Checkpoint 10: sanity-check the normalisation

Two quick tests. **Left:** TPM column sums should all be exactly 1,000,000. **Right:** log TPM distributions per sample should overlay each other closely; one curve far off from the rest is an outlier worth investigating.

In [ ]:
if lengths is not None:
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].hist(tpm.sum(), bins=20, color="#4C78A8")
    ax[0].set_title("TPM column sums (should be 1e6)")
    for s in tpm.columns[:60]:
        v = np.log2(tpm[s] + 1)
        ax[1].hist(v, bins=60, histtype="step", alpha=.4, density=True)
    ax[1].set_title("log2(TPM+1) per sample (first 60)")
    plt.tight_layout(); plt.show()
else:
    print("no gene lengths, nothing to plot")

---
# Part 11 · Write outputs & final audit

🎯 **Goal:** write the sample sheet and supporting CSVs, then audit the weakest matches.

💡 **Why these files:**

| File | Use |
|---|---|
| `rnaseq_samples.csv` | the **analysis cohort**: one row per kept sample, with demographics, no DOB, no raw dates |
| `rnaseq_counts_raw.csv` / `_rpkm.csv` / `_tpm.csv` | expression matrices, columns in the same order as the sample sheet |
| `unmatched_to_metadata.csv` | kept samples with **no** demographics to chase down |
| `rnaseq_samples_ALL_with_flags.csv` | **every** sample with every flag. This is your audit trail for why anything was excluded |
| `gene_lengths.csv` | cached so the GTF is parsed only once |

The last printout lists every kept sample whose match was *not* a strong ID match. Those are the ones to eyeball before trusting their demographics.

In [ ]:
# ---- 7. sample sheet (no birth date, no raw dates) -------------------------
cols = ["sample_id", "patient_key", "enrollment_id_rna", "initials_rna", "meta_join",
        "sample_source", "treatment", "tx_status", "diagnosis_rna", "dx_clean", "diagnosis1",
        "diagnosis2", "study_status", "sex", "race", "ethnicity", "age_yrs", "age_date_source",
        "sequencing_facility", "sequencing_cohort", "total_counts", "source_file"]
final_meta[cols].to_csv(os.path.join(OUT_DIR, "rnaseq_samples.csv"), index=False)
final_meta.loc[final_meta["meta_join"] == "none",
               ["sample_id", "enrollment_id_rna", "initials_rna", "dx_clean", "source_file"]
               ].to_csv(os.path.join(OUT_DIR, "unmatched_to_metadata.csv"), index=False)
rmeta.drop(columns=["_dob", "_date", "_md_enroll", "meta_row"]).to_csv(
    os.path.join(OUT_DIR, "rnaseq_samples_ALL_with_flags.csv"), index=False)
print("\nwritten to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

### Final audit: weak or unmatched kept samples

In [ ]:
k = final_meta
w = k[~k["meta_join"].isin(["enrollment_id", "R_prefix_id"])]
print(pd.crosstab(k["dx_clean"], k["meta_join"]).to_string())
print(f"\nweak / unmatched: {len(w)}")
print(w[["sample_id", "enrollment_id_rna", "initials_rna", "meta_join", "patient_key", "dx_clean", "sex", "age_yrs"]]
      .assign(sample_id=w["sample_id"].str[:30]).to_string(index=False))

---
## ✅ Done

Suggested next steps:

1. Read the **Checkpoint 8** table and confirm the pre/post calls are right. This single regex decides cohort membership.
2. Work through the **weak / unmatched** list above; fix IDs in the metadata or source files and re-run.
3. Confirm the gene-length match rate in Part 10 is high (≳95%). If not, use a GTF whose release matches your counts.
4. Use `rnaseq_tpm.csv` for cross-sample comparisons and `rnaseq_counts_raw.csv` for count-based tools (DESeq2 / edgeR / WGCNA input).